# UdyamSetu Indic TTS API (Google Colab)

This notebook serves **Hindi (`hi`)**, **Marathi (`mr`)**, and **English (`en`)** speech with AI4Bharat Indic Parler-TTS. It starts an authenticated `POST /tts` API and creates a temporary public HTTPS URL with ngrok.

> Keep this Colab notebook private. A Colab runtime and its public URL stop when the runtime is disconnected or times out. For a production deployment, host the same FastAPI service on a persistent GPU service.

## Before running

1. In Colab, select **Runtime → Change runtime type → T4 GPU**.
2. Create a free ngrok account and copy its authtoken from the ngrok dashboard.
3. Run the cells in order.

In [1]:
# Install the model runtime and the small HTTP/tunnel server.
!pip -q install git+https://github.com/huggingface/parler-tts.git soundfile fastapi 'uvicorn[standard]' pyngrok nest-asyncio
!pip -q install --no-deps "protobuf>=5.28.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.0 MB/s eta 0:00:00
ERROR: pip's dependency reso

In [2]:
import os
import secrets
from google.colab import userdata

# Required to publish the temporary HTTPS URL. Do not commit this value.
os.environ['NGROK_AUTHTOKEN'] = userdata.get('NGROK_AUTHTOKEN')

# This is the API key your UdyamSetu backend will send in X-API-Key.
# Save it now; it is generated fresh each time the runtime starts.
TTS_API_KEY = secrets.token_urlsafe(32)
print('TTS API key (store this securely):', TTS_API_KEY)


TTS API key (store this securely): sPtlj5myAJuLulkTkml6s3ADuqwXEwt6nx6o6b_1pdk


In [3]:
import io
import re
import numpy as np
import torch
import soundfile as sf
from google.colab import userdata
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
from functools import lru_cache

# Retrieve token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

MODEL_ID = 'ai4b-hf/indic-parler-tts-pretrained-v3'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

if DEVICE == 'cpu':
    print('WARNING: No GPU found. Speech generation will be slow; use a Colab T4 GPU for the API.')

MODEL_DTYPE = torch.float16 if DEVICE.startswith('cuda') else torch.float32
if DEVICE.startswith('cuda'):
    torch.set_float32_matmul_precision('high')
model = ParlerTTSForConditionalGeneration.from_pretrained(
    MODEL_ID, token=hf_token, torch_dtype=MODEL_DTYPE
).to(DEVICE).eval()

prompt_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
description_tokenizer = AutoTokenizer.from_pretrained(
    model.config.text_encoder._name_or_path, token=hf_token
)

SAMPLE_RATE = model.config.sampling_rate
print(f'Model ready on {DEVICE}; sample rate: {SAMPLE_RATE} Hz')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.75G [00:00<?, ?B/s]

  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

  "_name_or_path": "ylacombe/dac_44khz",
  "architectures": [
    "DacModel"
  ],
  "codebook_dim": 8,
  "codebook_loss_weight": 1.0,
  "codebook_size": 1024,
  "commitment_loss_weight": 0.25,
  "decoder_hidden_si

generation_config.json:   0%|          | 0.00/218 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/990 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Model ready on cuda:0; sample rate: 44100 Hz


In [4]:
# A language-specific description helps keep the generated voice natural and consistent.
VOICE_DESCRIPTIONS = {
    'en': 'Mary speaks Indian English in a warm, clear, helpful voice at a moderate pace. The recording is very high quality with no background noise.',
    'hi': 'Divya speaks Hindi in a warm, clear, helpful voice at a moderate pace. The recording is very high quality with no background noise.',
    'mr': 'Sunita speaks Marathi in a warm, clear, helpful voice at a moderate pace. The recording is very high quality with no background noise.',
}

@torch.inference_mode()
def _generate_segment(text: str, language: str):
    description = description_tokenizer(VOICE_DESCRIPTIONS[language], return_tensors='pt').to(DEVICE)
    prompt = prompt_tokenizer(text, return_tensors='pt').to(DEVICE)
    return model.generate(
        input_ids=description.input_ids,
        attention_mask=description.attention_mask,
        prompt_input_ids=prompt.input_ids,
        prompt_attention_mask=prompt.attention_mask,
        do_sample=True,
        temperature=1.0,
        min_new_tokens=32,
        max_new_tokens=2048,
    ).cpu().float().numpy().squeeze()

@lru_cache(maxsize=128)
def synthesize_wav(text: str, language: str) -> bytes:
    if language not in VOICE_DESCRIPTIONS:
        raise ValueError('language must be one of: en, hi, mr')
    text = text.strip()
    if not text:
        raise ValueError('text cannot be empty')
    if len(text) > 600:
        raise ValueError('text is limited to 600 characters per request')

    # Indic Parler-TTS detects the spoken language from `text`. Hindi/Marathi must be sent in Devanagari, not Latin transliteration.
    # Generate each sentence separately so an early EOS token cannot drop later sentences.
    segments = [part.strip() for part in re.split(r'(?<=[.!?।])\s+', text) if part.strip()]
    audio = np.concatenate([_generate_segment(segment, language) for segment in segments])
    buffer = io.BytesIO()
    sf.write(buffer, audio, SAMPLE_RATE, format='WAV', subtype='PCM_16')
    buffer.seek(0)
    return buffer.read()

# Quick local test before exposing the endpoint.
test_audio = synthesize_wav('नमस्कार! मी उद्यमसेतू सहाय्यक आहे. तुमच्या व्यवसायासाठी योग्य योजना शोधण्यात मी मदत करेन.', 'mr')
from IPython.display import Audio, display
display(Audio(test_audio, rate=SAMPLE_RATE))


In [5]:
import threading
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel, Field
from pyngrok import ngrok

app = FastAPI(title='UdyamSetu Indic TTS', version='1.0')

class TTSRequest(BaseModel):
    text: str = Field(min_length=1, max_length=600)
    language: str = Field(pattern='^(en|hi|mr)$')

def authorize(x_api_key: str | None):
    if not x_api_key or not secrets.compare_digest(x_api_key, TTS_API_KEY):
        raise HTTPException(status_code=401, detail='Invalid API key')

@app.get('/health')
def health():
    return {'status': 'ok', 'languages': ['en', 'hi', 'mr']}

@app.post('/tts')
def text_to_speech(payload: TTSRequest, x_api_key: str | None = Header(default=None)):
    authorize(x_api_key)
    try:
        wav = synthesize_wav(payload.text, payload.language)
        return Response(content=wav, media_type='audio/wav', headers={'Cache-Control': 'private, max-age=86400'})
    except ValueError as error:
        raise HTTPException(status_code=400, detail=str(error))
    except Exception as error:
        print(f'TTS generation error: {type(error).__name__}: {error}')
        raise HTTPException(status_code=503, detail='TTS generation temporarily unavailable')

nest_asyncio.apply()
server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8001, log_level='warning'))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
tunnel = ngrok.connect(8001, bind_tls=True)
PUBLIC_TTS_URL = tunnel.public_url
print('Public TTS endpoint:', f'{PUBLIC_TTS_URL}/tts')
print('Health endpoint:', f'{PUBLIC_TTS_URL}/health')
print('\nAdd these to the repository-root .env file (never commit the API key):')
print(f'INDIC_TTS_URL={PUBLIC_TTS_URL}/tts')
print(f'INDIC_TTS_API_KEY={TTS_API_KEY}')


Public TTS endpoint: https://radishlike-overimpressionable-nikita.ngrok-free.dev/tts
Health endpoint: https://radishlike-overimpressionable-nikita.ngrok-free.dev/health

Add these to the repository-root .env file (never commit the API key):
INDIC_TTS_URL=https://radishlike-overimpressionable-nikita.ngrok-free.dev/tts
INDIC_TTS_API_KEY=sPtlj5myAJuLulkTkml6s3ADuqwXEwt6nx6o6b_1pdk


In [6]:
# Test the public API exactly as your UdyamSetu backend will call it.
import requests

response = requests.post(
    f'{PUBLIC_TTS_URL}/tts',
    headers={'X-API-Key': TTS_API_KEY},
    json={'text': 'नमस्कार! मी उद्यमसेतू सहाय्यक आहे.', 'language': 'mr'},
    timeout=180,
)
response.raise_for_status()
display(Audio(response.content, rate=SAMPLE_RATE))


## API contract

```http
POST {INDIC_TTS_URL}
X-API-Key: {INDIC_TTS_API_KEY}
Content-Type: application/json

{"text": "आपकी सहायता के लिए मैं यहाँ हूँ।", "language": "hi"}
```

The response body is `audio/wav`. Add the printed values to the repository-root `.env` file (next to `.env.example`). The UdyamSetu backend should proxy that audio to the frontend; do not expose `INDIC_TTS_API_KEY` to the browser.

### Operational notes

- The generated URL is temporary and changes after a Colab restart. Update `INDIC_TTS_URL` whenever it changes.
- Colab is suitable for demos, not production: it can disconnect and has limited concurrency.
- Keep text short (1–3 sentences) for lower latency. The endpoint enforces a 600-character request limit.
- Your chatbot must send Devanagari Hindi/Marathi text for best pronunciation, not Latin transliterations.